In [1]:
# ### Cell 1: Imports and Setup ###

import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate

# Load the API key from the .env file
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# Check if the API key is available
if not api_key:
    raise ValueError("Google API Key not found. Please set it in the .env file.")
else:
    print("Libraries imported and API key loaded successfully.")

Libraries imported and API key loaded successfully.


In [6]:
# ### Cell 2: Create the Insight Chain ###

# Initialize the Gemini LLM
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest", google_api_key=api_key, temperature=0.5)

# Define the prompt template from above
insight_prompt_template = """
You are a highly skilled medical analyst. Your task is to read the following medical document and extract the top 3-5 most critical insights for a general audience.
**Address the user directly in the second person (e.g., "your results show," "you should consider"). Do not refer to "the patient."**

An "insight" can be:
- A surprising statistic or data point.
- An actionable recommendation or key takeaway.
- The main conclusion or finding of the document.
- An important, non-obvious connection between different pieces of information.

Present the insights clearly, using bullet points. Do not summarize the entire document; focus only on the most crucial pieces of information.

Document Text:
---
{document_text}
---

Top Insights:
"""

# Create the prompt and the LLMChain
prompt = PromptTemplate(template=insight_prompt_template, input_variables=["document_text"])
insight_chain = LLMChain(llm=llm, prompt=prompt)

print("Insight generation chain is ready.")

Insight generation chain is ready.


In [7]:
# ### Cell 3: Run and Get Insights ###

# Specify the path to the document you want to analyze
pdf_path = "./med_report/test_report.pdf" # <-- CHANGE THIS to your desired file

print(f"Loading document: {pdf_path}")
loader = PyPDFLoader(pdf_path)
docs = loader.load()

# Combine the content of all pages into a single text block
full_document_text = "\n".join(doc.page_content for doc in docs)

print("\nGenerating insights")
# Run the chain on the full document text
insights_result = insight_chain.invoke({"document_text": full_document_text})

# Print the final insights
print("-" * 50)
print("Key Insights from the Document:")
print("-" * 50)
print(insights_result['text'])

Loading document: ./med_report/test_report.pdf

Generating insights
--------------------------------------------------
Key Insights from the Document:
--------------------------------------------------
* Your hemoglobin level is low (12.5 g/dL), which falls outside the normal range (13.0-17.0 g/dL), indicating a possible anemia.  This requires further investigation.

* Your platelet count is borderline (150,000 cumm) at the lower end of the normal range (150,000-410,000 cumm).  While currently within the normal range, it warrants monitoring for potential future issues.

* Your packed cell volume (PCV) is high (57.5%), exceeding the normal range (40-50%). This, combined with your low hemoglobin, suggests a need for further testing to determine the cause of this discrepancy and to clarify the anemia diagnosis.  The lab report specifically recommends confirming this anemia diagnosis.
